# Totales por circuito

Un JSON por (año, nivel) que unifica, por `circuito_id`, los totales de cada agrupación y de los "otros" (EN BLANCO, NULO, RECURRIDO, IMPUGNADO). Un archivo por nivel (no compartido entre niveles) para que no se pisen entre sí.

Output: `data/<año>/<nivel>/circuito_<nivel>.json`, junto al `.json` (agregado) y `.csv` (oficial) que ya había en esa carpeta:

```
{
  "anio": 2011,
  "nivel": "intendente",
  "categoria_id": 7,
  "circuitos": {
    "0460": {
      "mesas": 6,
      "electores": 2088,
      "positivos": {"0047": {"nombre": "...", "votos": 123, "campo_ideologico": "4"}, ...},
      "otros": {"EN BLANCO": 45, "NULO": 12, "RECURRIDO": 3, "IMPUGNADO": 1}
    },
    ...
  }
}
```

`campo_ideologico` se copia tal cual de `data/agrupaciones/agrupaciones.csv` / `agrupaciones_legislativas.csv`

In [ ]:
import csv
import io
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

from electoral.client import ResultadosClient

REPO = Path.cwd().parent
client = ResultadosClient(cache_dir=REPO / "data")

LA_PLATA = dict(
    tipo_eleccion=2,  # Generales
    distrito_id=2,  # Buenos Aires
    seccion_provincial_id=8,  # Sección Capital
    seccion_id=63,  # La Plata
)

## 1. Función de agregación por circuito

A partir del CSV oficial (ya en caché), agrupa por `circuito_id` sumando `votos_cantidad`: por agrupación para las filas `POSITIVO`, y por `votos_tipo` para el resto.

In [ ]:
def agregar_por_circuito(df):
    df = df.copy()
    df["circuito_id"] = df["circuito_id"].str.strip()  # el CSV trae algunos circuito_id con espacio de relleno
    df["agrupacion_nombre"] = df["agrupacion_nombre"].str.strip()

    circuitos = {}
    for circuito_id, grupo in df.groupby("circuito_id"):
        positivos = {}
        for agrupacion_id, sub in grupo[grupo["votos_tipo"] == "POSITIVO"].groupby("agrupacion_id"):
            positivos[str(agrupacion_id)] = {
                "nombre": sub["agrupacion_nombre"].iloc[0],
                "votos": int(sub["votos_cantidad"].sum()),
            }

        otros = (
            grupo[grupo["votos_tipo"] != "POSITIVO"]
            .groupby("votos_tipo")["votos_cantidad"]
            .sum()
            .astype(int)
            .to_dict()
        )

        circuitos[circuito_id] = {
            "mesas": int(grupo["mesa_id"].nunique()),
            "electores": int(grupo.drop_duplicates("mesa_id")["mesa_electores"].sum()),
            "positivos": positivos,
            "otros": otros,
        }
    return circuitos

## 1.1 Campo ideológico

Cada agrupación en `positivos` suma el campo `campo_ideologico`, copiado tal cual está en `data/agrupaciones/agrupaciones.csv` / `agrupaciones_legislativas.csv`. El join es por (`anio`, `nivel`, `agrupacion`) exacto.

Si una agrupación no aparece en la clasificación, esto tiene que fallar (`KeyError`) en vez de guardar el circuito sin el campo — no hay valor "sin clasificar" implícito.

In [ ]:
NIVEL_A_NIVEL_CSV = {"gobernador": "gobernacion"}


def cargar_clasificacion():
    filas = []
    for nombre in ["agrupaciones.csv", "agrupaciones_legislativas.csv"]:
        with open(REPO / "data" / "agrupaciones" / nombre, encoding="utf-8") as f:
            filas.extend(csv.DictReader(f))
    return {(r["anio"], r["nivel"], r["agrupacion"]): r["campo_ideologico"] for r in filas}


CLASIFICACION = cargar_clasificacion()


def agregar_campo_ideologico(circuitos, anio, nivel):
    nivel_csv = NIVEL_A_NIVEL_CSV.get(nivel, nivel)
    for c in circuitos.values():
        for info in c["positivos"].values():
            info["campo_ideologico"] = CLASIFICACION[(str(anio), nivel_csv, info["nombre"])]
    return circuitos

## 2. Caso de referencia: traer el CSV y aplicar la agregación (2011 / Intendente)

In [ ]:
ANIO = 2011
NIVEL = "intendente"
CATEGORIA_ID = 7

csv_bytes = client.get_resultados_csv(
    anio_eleccion=ANIO, categoria_nombre=NIVEL, categoria_id=CATEGORIA_ID, **LA_PLATA
)
df = pd.read_csv(io.BytesIO(csv_bytes), low_memory=False)

circuitos = agregar_por_circuito(df)
circuitos = agregar_campo_ideologico(circuitos, ANIO, NIVEL)
print(f"circuitos: {len(circuitos)}")
primero = next(iter(circuitos))
print(f"ejemplo ({primero}):", json.dumps(circuitos[primero], indent=2, ensure_ascii=False))

## 3. Validar contra el agregado de la sección

Sumar todos los circuitos tiene que dar exactamente el agregado de toda La Plata para esta categoría (ya sabemos, de los notebooks anteriores, que ese agregado es confiable para 2011/Intendente).

In [ ]:
raw_agregado = client.get_resultados(
    anio_eleccion=ANIO, categoria_nombre=NIVEL, categoria_id=CATEGORIA_ID, **LA_PLATA
)


def normalizar_id(x):
    x = str(x)
    return (x.lstrip("0") or "0") if x.isdigit() else x


positivos_circuitos = {}
otros_circuitos = {}
mesas_total = 0
electores_total = 0
for c in circuitos.values():
    mesas_total += c["mesas"]
    electores_total += c["electores"]
    for agrupacion_id, info in c["positivos"].items():
        clave = normalizar_id(agrupacion_id)
        positivos_circuitos[clave] = positivos_circuitos.get(clave, 0) + info["votos"]
    for tipo, votos in c["otros"].items():
        otros_circuitos[tipo] = otros_circuitos.get(tipo, 0) + votos

positivos_agregado = {
    normalizar_id(a["idAgrupacion"]): a["votos"] for a in raw_agregado["valoresTotalizadosPositivos"]
}

otros_total_circuitos = sum(otros_circuitos.values())
otros_total_agregado = (
    raw_agregado["valoresTotalizadosOtros"]["votosNulos"]
    + raw_agregado["valoresTotalizadosOtros"]["votosEnBlanco"]
    + raw_agregado["valoresTotalizadosOtros"]["votosRecurridosComandoImpugnados"]
)

print("categorías de 'otros' encontradas en este año:", sorted(otros_circuitos))
print("positivos: circuitos vs agregado ->",
      "OK" if positivos_circuitos == positivos_agregado else f"DIFERENCIA: {positivos_circuitos} vs {positivos_agregado}")
print("otros (total): circuitos =", otros_total_circuitos, "vs agregado =", otros_total_agregado)
print("mesas:", mesas_total, "vs", raw_agregado["estadoRecuento"]["mesasTotalizadas"])
print("electores:", electores_total, "vs", raw_agregado["estadoRecuento"]["cantidadElectores"])

assert positivos_circuitos == positivos_agregado
assert otros_total_circuitos == otros_total_agregado
assert mesas_total == raw_agregado["estadoRecuento"]["mesasTotalizadas"]
assert electores_total == raw_agregado["estadoRecuento"]["cantidadElectores"]
print("\nOK: la suma por circuito coincide exactamente con el agregado de la sección.")

## 4. Guardar `data/<año>/<nivel>/circuito_<nivel>.json`

In [ ]:
destino = REPO / "data" / str(ANIO) / NIVEL / f"circuito_{NIVEL}.json"

contenido = {"anio": ANIO, "nivel": NIVEL, "categoria_id": CATEGORIA_ID, "circuitos": circuitos}

destino.write_text(json.dumps(contenido, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"{len(circuitos)} circuitos -> {destino}")

## 5. Función reutilizable + batch para todos los (año, nivel)

Empaqueta los pasos 2-4 en una función, con la misma validación pero sin `assert` (para que un caso con diferencia no frene los demás — solo se reporta).

In [ ]:
def normalizar_id(x):
    x = str(x)
    return (x.lstrip("0") or "0") if x.isdigit() else x


def procesar(anio, nivel, categoria_id):
    csv_bytes = client.get_resultados_csv(
        anio_eleccion=anio, categoria_nombre=nivel, categoria_id=categoria_id, **LA_PLATA
    )
    df = pd.read_csv(io.BytesIO(csv_bytes), low_memory=False)
    circuitos = agregar_por_circuito(df)
    circuitos = agregar_campo_ideologico(circuitos, anio, nivel)

    raw_agregado = client.get_resultados(
        anio_eleccion=anio, categoria_nombre=nivel, categoria_id=categoria_id, **LA_PLATA
    )

    positivos_circuitos, otros_circuitos = {}, {}
    mesas_total = electores_total = 0
    for c in circuitos.values():
        mesas_total += c["mesas"]
        electores_total += c["electores"]
        for agrupacion_id, info in c["positivos"].items():
            clave = normalizar_id(agrupacion_id)
            positivos_circuitos[clave] = positivos_circuitos.get(clave, 0) + info["votos"]
        for tipo, votos in c["otros"].items():
            otros_circuitos[tipo] = otros_circuitos.get(tipo, 0) + votos

    positivos_agregado = {
        normalizar_id(a["idAgrupacion"]): a["votos"] for a in raw_agregado["valoresTotalizadosPositivos"]
    }
    # "otros" se compara por total, no por nombre de categoría: votos_tipo no es estable
    # entre años (NULO/NULOS, EN BLANCO/BLANCOS...) y desde 2019 aparece COMANDO, que no
    # existía en 2011. agregar_por_circuito ya agrupa por lo que sea que diga votos_tipo,
    # así que el total siempre está bien igual — solo la comparación por nombre fallaría.
    otros_total_circuitos = sum(otros_circuitos.values())
    otros_total_agregado = (
        raw_agregado["valoresTotalizadosOtros"]["votosNulos"]
        + raw_agregado["valoresTotalizadosOtros"]["votosEnBlanco"]
        + raw_agregado["valoresTotalizadosOtros"]["votosRecurridosComandoImpugnados"]
    )

    ok = (
        positivos_circuitos == positivos_agregado
        and otros_total_circuitos == otros_total_agregado
        and mesas_total == raw_agregado["estadoRecuento"]["mesasTotalizadas"]
        and electores_total == raw_agregado["estadoRecuento"]["cantidadElectores"]
    )

    destino = REPO / "data" / str(anio) / nivel / f"circuito_{nivel}.json"
    contenido = {"anio": anio, "nivel": nivel, "categoria_id": categoria_id, "circuitos": circuitos}
    destino.write_text(json.dumps(contenido, indent=2, ensure_ascii=False), encoding="utf-8")

    return {
        "anio": anio,
        "nivel": nivel,
        "categoria_id": categoria_id,
        "ok": ok,
        "circuitos": len(circuitos),
        "destino": str(destino),
        "otros_categorias": sorted(otros_circuitos),
    }

In [ ]:
EXECUTIVOS = [
    (anio, nivel, categoria_id)
    for anio in [2011, 2015, 2019, 2023]
    for nivel, categoria_id in [("presidente", 1), ("gobernador", 4), ("intendente", 7)]
]

# nacional usa Diputados Nacionales (idCargo=3, consistente los 4 años) — ver aviso arriba
# sobre Senador Nacional (idCargo=2, solo 2017), que queda afuera.
LEGISLATIVOS = [
    (2013, "nacional", 3), (2013, "provincial", 6), (2013, "municipal", 10),
    (2017, "nacional", 3), (2017, "provincial", 6), (2017, "municipal", 10),
    (2021, "nacional", 3), (2021, "provincial", 6), (2021, "municipal", 10),
    (2025, "nacional", 3),  # provincial/municipal no disponibles en 2025 (hallazgo #12)
]

resultados = []
for anio, nivel, categoria_id in EXECUTIVOS + LEGISLATIVOS:
    r = procesar(anio, nivel, categoria_id)
    resultados.append(r)
    estado = "OK" if r["ok"] else "DIFERENCIA vs agregado JSON"
    print(f"{anio}/{nivel} (idCargo={categoria_id}): {r['circuitos']} circuitos, {estado}")

no_ok = [r for r in resultados if not r["ok"]]
print(f"\n{len(resultados)} archivos generados. {len(no_ok)} con diferencia contra el agregado JSON "
      f"(esperado si ese agregado ya era conocido como no confiable):")
for r in no_ok:
    print(f"  {r['anio']}/{r['nivel']}")